# CyteOnto backup pair: ontology descriptions and embeddings

Builds the **secondary (fallback) model pair** ontology cache under `cyteonto/data/`:

| Role | Provider | Model |
|------|----------|-------|
| Descriptions | `fireworks` | `accounts/fireworks/models/kimi-k2p6` |
| Embeddings | `openrouter` | `Qwen/Qwen3-Embedding-8B` (DeepInfra routing) |

This is separate from the **primary** pair (`nebius` / `nebius`) used by `setup.py` and the quick tutorial. Files are keyed by provider and model name (schema v3).

Requires in `.env`:

- `FIREWORKS_API_KEY`
- `OPENROUTER_API_KEY`

Optional: `NCBI_API_KEY` if `use_pubmed_tool=True`.

The first full generation can take a long time (one LLM call per Cell Ontology term, then embedding). Shipped CSV and OWL must already exist (`uv run python cyteonto/setup.py` downloads those only).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

In [ ]:
DATA_DIR = REPO_ROOT / "cyteonto" / "data"

In [ ]:
import os

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from cyteonto import CyteOnto, EmbdConfig, LlmConfig
from cyteonto.config import Config
from cyteonto.paths import PathConfig, artifact_key_segment

In [ ]:
cfg = Config()

LLM_BASE_URLS = {
    "fireworks": "https://api.fireworks.ai/inference/v1",
    "openrouter": "https://openrouter.ai/api/v1/",
}


def build_agent(provider: str, model: str) -> Agent:
    env_var = cfg.PROVIDER_API_KEY_ENV[provider]
    return Agent(
        OpenAIChatModel(
            model,
            provider=OpenAIProvider(
                api_key=os.environ[env_var],
                base_url=LLM_BASE_URLS[provider],
            ),
        )
    )


In [ ]:
backup_agent = build_agent(
    cfg.FALLBACK_LLM_PROVIDER,
    cfg.FALLBACK_LLM_MODEL
)

backup_embedding = EmbdConfig(
    provider=cfg.FALLBACK_EMBEDDING_PROVIDER,
    model=cfg.FALLBACK_EMBEDDING_MODEL,
    apiKey=os.environ[cfg.PROVIDER_API_KEY_ENV[cfg.FALLBACK_EMBEDDING_PROVIDER]],
)

backup_llm = LlmConfig(
    provider=cfg.FALLBACK_LLM_PROVIDER,
    model=cfg.FALLBACK_LLM_MODEL,
)

backup_llm_key = backup_llm.to_artifact_key()
backup_embd_key = backup_embedding.to_artifact_key()

## Target paths

Ontology descriptions and embeddings are written here (same tree as the primary pair, different filename segments).

In [ ]:
paths = PathConfig(data_dir=DATA_DIR)

desc_path = paths.ontology_descriptions(backup_llm_key)
emb_path = paths.ontology_embeddings(backup_llm_key, backup_embd_key)

print("LLM segment:", artifact_key_segment(backup_llm_key))
print("Embedding segment:", artifact_key_segment(backup_embd_key))
print("Descriptions:", desc_path)
print("Embeddings:", emb_path)
print("Descriptions exist:", desc_path.exists())
print("Embeddings exist:", emb_path.exists())

## Optional: download from R2

If precomputed backup artifacts are already on the CDN, download them instead of calling the APIs. Skip this cell when you want a fresh local generation.

In [ ]:
import requests

BASE_URL = "https://pub-d8bf3af01ebe421abded39c4cb33d88a.r2.dev/cyteonto_v2"

DOWNLOAD_URLS = {
    desc_path: (
        f"{BASE_URL}/descriptions/descriptions_"
        f"{artifact_key_segment(backup_llm_key)}.json"
    ),
    emb_path: (
        f"{BASE_URL}/embeddings/embeddings_"
        f"{artifact_key_segment(backup_llm_key)}_"
        f"{artifact_key_segment(backup_embd_key)}.npz"
    ),
}

TRY_DOWNLOAD_FROM_R2 = False  # set True to attempt CDN download first

if TRY_DOWNLOAD_FROM_R2:
    for dest, url in DOWNLOAD_URLS.items():
        if dest.exists():
            print(f"skip (exists): {dest.name}")
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        resp = requests.get(url, timeout=120)
        if resp.status_code == 404:
            print(f"not on CDN: {dest.name}")
            continue
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        print(f"downloaded: {dest}")
else:
    print("R2 download disabled; will use from_config below.")

## Generate descriptions and embeddings

`CyteOnto.from_config` fills missing CL term descriptions, then builds the ontology embedding matrix.

Set `FORCE_REGENERATE = True` to delete existing backup artifacts and rebuild from scratch.

In [ ]:
FORCE_REGENERATE = True

cyto = await CyteOnto.from_config(
    agent=backup_agent,
    embedding=backup_embedding,
    llm=backup_llm,
    data_dir=DATA_DIR,
    force_regenerate=FORCE_REGENERATE,
    use_pubmed_tool=False,  # bool(os.getenv("NCBI_API_KEY")),
)

## Verify artifacts

In [ ]:
from cyteonto import storage

assert desc_path.exists(), f"missing descriptions: {desc_path}"
assert emb_path.exists(), f"missing embeddings: {emb_path}"

descriptions = storage.load_descriptions(desc_path)
assert descriptions is not None
loaded = storage.load_ontology_embeddings(emb_path)
assert loaded is not None
embeddings, ontology_ids, meta = loaded

non_blank = sum(1 for d in descriptions.values() if not d.is_blank())

print(f"Descriptions: {len(descriptions)} terms ({non_blank} non-blank)")
print(f"Embeddings: shape {embeddings.shape}, {len(ontology_ids)} ids")
print(f"NPZ metadata schema: {meta.get('schemaVersion')}")
print(f"LLM usage: {cyto.usage.model_dump()}")
print(f"Descriptions file size (MB): {desc_path.stat().st_size / 1e6:.2f}")
print(f"Embeddings file size (MB): {emb_path.stat().st_size / 1e6:.2f}")

## Use the backup pair for comparisons (optional)

Pass the same `agent`, `embedding`, and `llm` to `compare`. Do not pass primary agents as `fallback_*` unless you want failover during the run.

In [ ]:
df = await cyto.compare(
    author_labels=["animal stem cell", "BFU-E"],
    algorithms={"method_a": ["stem cell", "blast forming unit erythroid"]},
    run_id="backup_pair",
)

In [ ]:
df